In [ ]:
"""duckdb==1.0.0
ipykernel==6.29.4
pandas==2.2.2
pyarrow==16.1.0
boto3==1.34.118
fastavro==1.9.4
matplotlib==3.9.0
s3fs==2024.6.0"""
!pip install duckdb
!pip install ipykernel
!pip install pandas
!pip install pyarrow
!pip install boto3
!pip install fastavro
!pip install matplotlib
!pip install s3fs

In [ ]:
import pandas as pd
import time
import boto3
from io import StringIO, BytesIO
import matplotlib.pyplot as plt
from fastavro import writer, parse_schema, reader
import pyarrow.parquet as pq
import pyarrow as pa
import datetime as dt

# Serialización

Vamos q analizar diferentes formatos de datos. Para esto los vamos a escribir y leer de un almacenamiento distribuido usando [minio](https://min.io/) un FOSS object store compatible con S3.

Estos formatos se pueden usar tanto para persistir los datos en el master dataset de una arquitectura lambda como para transferir información en su speed layer.

# Crear el bucket y traer los datos de prueba

Corriendo run_minio.sh se va a levantar un contenedor de docker con minio. Va a bindear el puerto 9000 a la api, y en el puerto 9090 podemos acceder a la ui web. Los datos se van a persistir en un volumen (./minio_data).

In [ ]:
# https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html
s3 = boto3.client('s3',
                  endpoint_url='http://localhost:9000',
                  aws_access_key_id='catedra',
                  aws_secret_access_key='catedrapass')

In [ ]:
fvehicular_bucket = 'flujovehicular'
if fvehicular_bucket not in [b["Name"] for b in s3.list_buckets()['Buckets']]:
    s3.create_bucket(Bucket=fvehicular_bucket)

In [ ]:
DATASET_URL = "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/transporte-y-obras-publicas/flujo-vehicular-anillo-digital/dataset_flujo_vehicular.csv"
df = pd.read_csv(DATASET_URL)
df["HORA"] = pd.to_datetime(df["HORA"], format="%d%b%Y:%H:%M:%S")
df.head()

# CSV

 - Archivo de **texto**
 - Orientado a filas
 - No tiene un esquema definido
 - Mo soporta cambios en el esquema 
 - Los datos están crudos
 - Sólo soporta tipos de datos primitivos
 - No contempla bloques para su organización
 - Tenés que leer todo el archivo para hacer consultas
 - Puede agregar filas a archivos existentes eficientemente
 - Permite la inspeccion directa de los datos
 - Sirve para trabajar con conjuntos de datos que entran en memoria

In [ ]:
start_time = time.time()
with StringIO() as csv_buffer:
    df.to_csv(csv_buffer, index=False)
    s3.put_object(Body=csv_buffer.getvalue(), Bucket=fvehicular_bucket, Key='flujo.csv')
csv_write_time = time.time()-start_time
csv_write_time

In [ ]:
start_time = time.time()
csv_object = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.csv',)
df = pd.read_csv(csv_object["Body"], parse_dates=['HORA'])
csv_read_time = time.time() -start_time
csv_read_time

In [ ]:
csv_size = csv_object["ContentLength"]
csv_size

In [ ]:
csv_object_body = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.csv',)["Body"]
print(next(csv_object_body).decode())

In [ ]:
start_time = time.time()
csv_object = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.csv',)
df = pd.read_csv(csv_object["Body"], parse_dates=['HORA'])
result_df = df.query("CANTIDAD > 13000 & SENTIDO=='Egreso'")
csv_query_time = time.time() -start_time
csv_query_time

In [ ]:
result_df

In [ ]:
start_time = time.time()
csv_object = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.csv',)
df = pd.read_csv(csv_object["Body"], parse_dates=['HORA'])
result_seq_df = df.query("HORA > '2020-03-10 00:00' & HORA < '2020-03-12 00:00'")
csv_query_seq_time = time.time() -start_time
csv_query_seq_time

In [ ]:
len(result_seq_df)

# AVRO

 - Archivo **binario**
 - Orientado a filas
 - Permite definir esquemas
 - Soporta evolucion del esquema
 - Ofrece opciones de compresión
 - Permite definir tipos de datos
 - Organiza los registros en bloques
 - Permite _predicate pushdown_
 - Puede agregar filas a archivos existentes eficientemente
 - Se usa en _streams_ como kafka o sistemas de almacenamiento distribuido como hadoop o s3
 - Su uso no esta pensado para trabajar sobre un solo archivo
 - Se enfoca en el intercambio de datos entre sistemas

In [ ]:
schema = {
    'name': 'flujo_vehicular',
    'type': 'record',
    'fields': [
       {'name': 'HORA', 'type': 'int'},
       {'name': 'CODIGO_LOCACION', 'type': 'string'},
       {'name': 'CANTIDAD', 'type': 'int'},
       {'name': 'SENTIDO', 'type': 'string'},
       {'name': 'LATITUD', 'type': 'float'},
       {'name': 'LONGITUD', 'type': 'float'}
    ]
}
parsed_schema = parse_schema(schema)

In [ ]:
df_avro = df.copy()
df_avro["HORA"] = (df["HORA"] - dt.datetime(1970,1,1)).dt.total_seconds().astype(int)
records = df_avro.sort_values(by=['HORA']).to_dict('records')

In [ ]:
start_time = time.time()
with BytesIO() as avro_buffer:
    writer(avro_buffer, parsed_schema, records)
    s3.put_object(Body=avro_buffer.getvalue(), Bucket=fvehicular_bucket, Key='flujo.avro')
avro_write_time = time.time()-start_time
avro_write_time

In [ ]:
start_time = time.time()
sb = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.avro')["Body"]
avro_records = list(reader(sb))
avro_read_time = time.time() -start_time
avro_read_time

In [ ]:
avro_size = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.avro')["ContentLength"]
avro_size

In [ ]:
csv_object_body = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.avro')["Body"]
print(next(csv_object_body))

In [ ]:
start_time = time.time()
result_records = []
sb = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.avro')["Body"]
for record in reader(sb):
    if record["CANTIDAD"] > 13000 and record["SENTIDO"]=="Egreso":
        result_records.append(record)
avro_query_time = time.time() -start_time
avro_query_time

In [ ]:
result_records

In [ ]:
start_date = int((dt.datetime(2020,3,10) - dt.datetime(1970,1,1)).total_seconds())
end_date = int((dt.datetime(2020,3,12) - dt.datetime(1970,1,1)).total_seconds())

start_time = time.time()
result_seq_records = []
sb = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.avro')["Body"]
# Se puede usar el orden secuencial de las filas para la búsqueda
# dejando de traer chunks cuando ya no sea necesario.
for record in reader(sb): 
    if record["HORA"] >= end_date:
        break
    if record["HORA"] > start_date:
        result_seq_records.append(record)
avro_query_seq_time = time.time() -start_time
avro_query_seq_time

In [ ]:
len(result_seq_records)

# Parquet

 - Archivo **binario**
 - Orientado a columnas
 - Permite definir esquemas
 - Soporta evolución del esquema
 - Ofrece opciones de compresión
 - Permite definir tipos de datos
 - Organiza los registros en bloques
 - Permite _predicate pushdown_
 - Permite proyectar sobre columnas rápidamente
 - Se usa en sistemas de almacenamiento distirbuido como hadoop o s3
 - Procesamiento distribuido
 - Su uso se enfoca en resolver consultas analíticas

In [ ]:
schema = pa.schema([
    ("CODIGO_LOCACION", pa.string()),
    ("HORA", pa.timestamp('ns')),
    ("CANTIDAD", pa.int32()),
    ("SENTIDO", pa.string()),
    ("LATITUD", pa.float32()),
    ("LONGITUD", pa.float32())
])

flujo_parq = pa.Table.from_pandas(df, schema=schema)

In [ ]:
start_time = time.time()
with BytesIO() as parquet_buffer:
    pq.write_table(flujo_parq, parquet_buffer)
    s3.put_object(Body=parquet_buffer.getvalue(), Bucket=fvehicular_bucket, Key='flujo.parquet')
parquet_write_time = time.time()-start_time
parquet_write_time

In [ ]:
start_time = time.time()
with BytesIO() as data:
    s3.download_fileobj(fvehicular_bucket, 'flujo.parquet', data)
    flujo_parq = pq.read_table(data)
parquet_read_time = time.time() -start_time
parquet_read_time

In [ ]:
parquet_size = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.parquet',)["ContentLength"]
parquet_size

In [ ]:
parquet_object_body = s3.get_object(Bucket=fvehicular_bucket, Key='flujo.parquet',)["Body"]
print(next(parquet_object_body))

In [ ]:
start_time = time.time()
with BytesIO() as data:
    s3.download_fileobj(fvehicular_bucket, 'flujo.parquet', data)
    result_parq = pq.read_table(data,
                               filters=[
                          ("CANTIDAD", ">", 13000),
                          ("SENTIDO", "=", "Egreso"),
                      ])
parquet_query_time = time.time() -start_time
parquet_query_time

In [ ]:
result_parq

In [ ]:
start_time = time.time()
with BytesIO() as data:
    s3.download_fileobj(fvehicular_bucket, 'flujo.parquet', data)
    result_seq_parq = pq.read_table(data,
                               filters=[
                          ("HORA", ">", pa.scalar(dt.datetime.strptime('2020-03-10', "%Y-%m-%d"))),
                          ("HORA", "<", pa.scalar(dt.datetime.strptime('2020-03-12', "%Y-%m-%d"))),
                      ])
parquet_query_seq_time = time.time() -start_time
parquet_query_seq_time

In [ ]:
len(result_seq_parq)

# Comparación

In [ ]:
fig, ax = plt.subplots()

format = ['csv', 'avro', 'parquet']
sizes = [csv_size/1024**2, avro_size/1024**2, parquet_size/1024**2]
ax.set_ylabel('File size (Mb)')

ax.bar(format, sizes);

In [ ]:
fig, ax = plt.subplots()


ops = ['reads', 'writes', 'queries', 'seq_queries']
format_times = {
    'csv': [csv_read_time, csv_write_time, csv_query_time, csv_query_seq_time],
    'avro': [avro_read_time, avro_write_time, avro_query_time, avro_query_seq_time],
    'parquet': [parquet_read_time, parquet_write_time, parquet_query_time, parquet_query_seq_time],
}

x = list(range(len(ops)))
width = 0.25
multiplier = 0

for attribute, measurement in format_times.items():
    offset = width * multiplier
    rects = ax.bar([x_i + offset for x_i in x], measurement, width, label=attribute)
    multiplier += 1

ax.set_ylabel('Time (seg)')
ax.set_xticks([x_i + width for x_i in x], ops)
ax.legend(ncols=3);